In [ ]:
"""
Stadium Weather Processor
--------------------------
For each stadium CSV in the input folder:
  - Filters rows to June 10–28 (all available years)
  - Computes hourly averages for temperature and humidity using a
    ±1-hour (3-hour) rolling window across all years combined
  - Retains stadium metadata columns
  - Combines all stadiums into a single output CSV

Usage:
    python process_stadiums.py --input_dir ./stadiums --output ./stadium_hourly_averages.csv

Arguments:
    --input_dir   Folder containing one CSV per stadium (default: ./stadiums)
    --output      Path for the combined output CSV (default: ./stadium_hourly_averages.csv)
"""

import argparse
import glob
import os
import tqdm

import numpy as np
import pandas as pd


# ---------------------------------------------------------------------------
# Config
# ---------------------------------------------------------------------------


# ---------------------------------------------------------------------------
# Helpers
# ---------------------------------------------------------------------------
def load_stadium(path: str) -> pd.DataFrame:
    df = pd.read_csv(path, index_col=0)
    df["time"] = pd.to_datetime(df["time"])
    return df


def filter_window(df: pd.DataFrame, start=(6, 10), end=(6, 28)) -> pd.DataFrame:
    """Keep rows whose (month, day) falls between start and end (inclusive)."""

    month = df["time"].dt.month
    day   = df["time"].dt.day
    # Convert to a comparable integer like MMDD for easy range filtering
    mmdd = month * 100 + day
    start_mmdd = start[0] * 100 + start[1]
    end_mmdd   = end[0]   * 100 + end[1]

    mask = (mmdd >= start_mmdd) & (mmdd <= end_mmdd)
    return df[mask].copy()



def compute_hourly_averages(df: pd.DataFrame, value_cols: list, window_hours: int = 1) -> pd.DataFrame:

    records = []

    for (month, day, hour), _ in df.groupby(["month", "day", "hour"]):
        neighbours = {(hour - window_hours) % 24, hour, (hour + window_hours) % 24}

        subset = df[
            (df["month"] == month) &
            (df["day"]   == day)   &
            (df["hour"].isin(neighbours))
        ]

        if subset.empty:
            continue

        row = {
            "month": month,
            "day":   day,
            "hour":  hour,
        }
        for col in value_cols:
            row[col] = subset[col].mean()

        records.append(row)

    return pd.DataFrame(records).sort_values(["month","day","hour"]).reset_index(drop=True)

def attach_metadata(hourly: pd.DataFrame, df_source: pd.DataFrame, metadat_cols ) -> pd.DataFrame:
    """Attach constant metadata from the first row of the source dataframe."""
    for col in metadat_cols:
        if col in df_source.columns:
            hourly[col] = df_source[col].iloc[0]
    return hourly


# ---------------------------------------------------------------------------
# Main
# ---------------------------------------------------------------------------
def calculate_hourly_averages(input_dir: str, output_path: str) -> None:
    meta_data_cols = ["month", "day", "hour",
    "name", "city", "latitude", "longitude",
    "utc_offset_seconds", "timezone", "timezone_abbreviation", "elevation",
    ]
    value_cols = ["temperature_2m", "relative_humidity_2m", "apparent_temperature"]

    csv_files = glob.glob(os.path.join(input_dir, "*.csv"))
    if not csv_files:
        raise FileNotFoundError(f"No CSV files found in '{input_dir}'")

    print(f"Found {len(csv_files)} stadium file(s) in '{input_dir}'")

    all_hourly = []

    for path in tqdm.tqdm(csv_files):
        stadium_name = os.path.splitext(os.path.basename(path))[0]
        print(f"  Processing: {stadium_name}")

        df = load_stadium(path)

        # 1. Filter to June 10–28
        df_filtered = filter_window(df)
        df_filtered["month"] = df["time"].dt.month
        df_filtered["day"]  = df["time"].dt.day
        df_filtered["hour"] = df["time"].dt.hour
        if df_filtered.empty:
            print(f"    ⚠  No data in June 10–28 window — skipping.")
            continue

        years = sorted(df_filtered["time"].dt.year.unique())
        print(f"    Years found: {years}  |  Rows: {len(df_filtered)}")

        # 2. Compute rolling hourly averages (across all years combined)
        hourly = compute_hourly_averages(df_filtered, value_cols )

        # 3. Attach metadata
        hourly = attach_metadata(hourly, df, metadat_cols=meta_data_cols)

        all_hourly.append(hourly)

    if not all_hourly:
        raise ValueError("No valid data was processed from any stadium file.")

    combined = pd.concat(all_hourly, ignore_index=True)

    # Reorder columns: metadata first, then hour, then values
    print(combined.columns)
    meta_present = [c for c in meta_data_cols if c in combined.columns]
    col_order = meta_present +  value_cols
    combined = combined[col_order]

    combined.to_csv(output_path, index=False)
    print(f"\n✅  Combined CSV written to: {output_path}")
    print(f"   Rows: {len(combined)}  |  Stadiums: {combined['name'].nunique()}")



In [ ]:
stad_dir = "datasets/stadiums/stadiums_hourly"
stad_out = "datasets/stadiums/stadium_hourly_averages.csv"
calculate_hourly_averages(stad_dir, stad_out )

In [ ]:

def filter_window_with_hours(df: pd.DataFrame, start=(6, 10), end=(6, 28), hours= (7,21)) -> pd.DataFrame:
    """Keep rows whose (month, day) falls between start and end (inclusive)."""

    month = df["time"].dt.month
    day   = df["time"].dt.day
    hour = df["time"].dt.hour
    # Convert to a comparable integer like MMDD for easy range filtering
    mmdd = month * 100 + day
    start_mmdd = start[0] * 100 + start[1]
    end_mmdd   = end[0]   * 100 + end[1]

    mask = (mmdd >= start_mmdd) & (mmdd <= end_mmdd) & ( hour>=hours[0]) &  ( hour<=hours[1]) 
   


    return  df[mask].copy()


def caculate_camp_monthly_daytime_averages(input_dir: str, output_path: str) -> None:
    meta_data_cols = ["month", "orig_team",
    "name",  "latitude", "longitude",
    "utc_offset_seconds", "timezone", "timezone_abbreviation", "elevation",
    ]
    value_cols = ["temperature_2m", "relative_humidity_2m", "apparent_temperature"]
    

    csv_files = glob.glob(os.path.join(input_dir, "*.csv"))
    if not csv_files:
        raise FileNotFoundError(f"No CSV files found in '{input_dir}'")

    print(f"Found {len(csv_files)} stadium file(s) in '{input_dir}'")

    all_hourly = []

    for path in tqdm.tqdm(csv_files):
        stadium_name = os.path.splitext(os.path.basename(path))[0]
        print(f"  Processing: {stadium_name}")

        df = load_stadium(path)

        # 1. Filter to June 10–28
        df_filtered = filter_window(df)
        df_filtered["month"] = df["time"].dt.month
        df_filtered["day"]  = df["time"].dt.day
        df_filtered["hour"] = df["time"].dt.hour
        if df_filtered.empty:
            print(f"    ⚠  No data in June 10–28 window — skipping.")
            continue

        years = sorted(df_filtered["time"].dt.year.unique())
        print(f"    Years found: {years}  |  Rows: {len(df_filtered)}")

        # 2. Compute monthly_day_time_averages
        monthly = df_filtered.groupby('month')[value_cols].mean().reset_index()
        print("df cols", df.columns)
        print("meta data cols", meta_data_cols)
        # 3. Attach metadata
        monthly = attach_metadata(monthly, df, metadat_cols=meta_data_cols)
        print('monthly_cols', monthly.columns)
        all_hourly.append(monthly)
    if not all_hourly:
        raise ValueError("No valid data was processed from any stadium file.")

    combined = pd.concat(all_hourly, ignore_index=True)

    # Reorder columns: metadata first, then hour, then values
    print(combined.columns)
    meta_present = [c for c in meta_data_cols if c in combined.columns]
    col_order = meta_present +  value_cols
    combined = combined[col_order]

    combined.to_csv(output_path, index=False)
    print(f"\n✅  Combined CSV written to: {output_path}")
    print(f"   Rows: {len(combined)}  |  Stadiums: {combined['name'].nunique()}")

In [ ]:
# stad_dir = "datasets/camps/camps_hourly"
# stad_out = "datasets/camps/camps_daytime_averages.csv"
# caculate_camp_monthly_daytime_averages(stad_dir, stad_out )